# Subnational Data Merge Pipeline v3 (KEN + SOM)

**Changes from v2:**
- Added FLDAS data (33 climate/hydrology Z-score variables)
- Added Vegetation Indices (NDVI, EVI, LST, VCI, TCI, VHI)
- Added Climate Indices (NINO34, IOD_DMI, Western_V_Gradient, MEI_v2)

**Changes from v1:**
- Target countries: KEN + SOM only (ETH removed — no price data in source CSV)
- Price columns: `c_maize_fao`, `c_food_price_index`, `c_sorghum` (v1 used `c_maize` which was empty for KEN)
- Spatial join: per-country ISO3 filtering to prevent cross-boundary misassignment
- Quality check section added

**Files:**
- `v1_original`: `subnational_merged_v1_original.parquet` / `subnational_merge_notebook_v1_original.ipynb`
- `v2`: `subnational_merged_v2_KEN_SOM.parquet`
- `v3 (this)`: `subnational_merged_v3_KEN_SOM.parquet` / `subnational_merge_notebook.ipynb`

## 0. Setup

In [16]:
import os
import logging
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from rasterstats import zonal_stats
import glob
import difflib
from shapely.geometry import Point
import itertools

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Resolve DATA_DIR relative to notebook location (src/notebook/)
DATA_DIR = os.path.abspath(os.path.join(os.path.dirname("__file__"), "../../data"))
iso3_list = ['KEN', 'SOM']
boundary_dir = os.path.join(DATA_DIR, 'geoboundaries')

print(f"DATA_DIR: {DATA_DIR}")
print(f"Exists: {os.path.isdir(DATA_DIR)}")
print(f"Target countries: {iso3_list}")

DATA_DIR: /Users/halimjun/Coding_local/price_prediction_clean/data
Exists: True
Target countries: ['KEN', 'SOM']


## 1. Helper Functions

In [17]:
def spatial_join_points(df, gdf, lon_col, lat_col):
    """Spatially join point data to Admin2 polygons."""
    points = gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df[lon_col], df[lat_col]), crs="EPSG:4326"
    )
    if gdf.crs != points.crs:
        gdf = gdf.to_crs(points.crs)
    joined = gpd.sjoin(points, gdf, how="left", predicate="within")
    joined.rename(columns={'shapeName': 'admin2_canonical'}, inplace=True)
    return joined

def fuzzy_match_names(series, choices, threshold=0.8):
    """Map names to canonical choices using fuzzy matching (difflib)."""
    mapping = {}
    unique_names = series.dropna().unique()
    for name in unique_names:
        matches = difflib.get_close_matches(str(name), choices, n=1, cutoff=threshold)
        if matches:
            mapping[name] = matches[0]
    return mapping

def process_worldpop_population(country_gdf, raster_path, iso_code):
    """Compute population per Admin2 using zonal statistics on WorldPop raster."""
    logger.info(f"Processing WorldPop for {iso_code} with {len(country_gdf)} regions...")
    country_gdf = country_gdf[country_gdf['shapeISO'] == iso_code].copy()
    if country_gdf.empty:
        return pd.DataFrame()
    stats = zonal_stats(country_gdf, raster_path, stats="sum", all_touched=True)
    country_gdf['population'] = [s['sum'] for s in stats]
    return country_gdf[['shapeName', 'population', 'shapeISO']].copy()

def load_canonical_boundaries(iso3_list, boundary_dir):
    """Load Admin2 boundaries for specified countries."""
    gdfs = []
    for iso in iso3_list:
        path = os.path.join(boundary_dir, f'gb_{iso}_ADM2.geojson')
        if os.path.exists(path):
            gdf = gpd.read_file(path)
            # Ensure shapeISO is populated (some files have empty values)
            if 'shapeISO' not in gdf.columns:
                gdf['shapeISO'] = iso
            else:
                gdf['shapeISO'] = gdf['shapeISO'].replace('', np.nan).fillna(iso)
            gdf = gdf[['shapeName', 'shapeISO', 'geometry']]
            gdfs.append(gdf)
            logger.info(f"Loaded {iso}: {len(gdf)} admin2 regions")
        else:
            logger.error(f"Not found: {path}")
    return pd.concat(gdfs, ignore_index=True)

def create_master_skeleton(years, months, admin_gdf):
    """Create master DataFrame with all Year x Month x Admin2 combinations."""
    records = []
    regions = admin_gdf[['shapeName', 'shapeISO']].drop_duplicates()
    for year in years:
        for month in months:
            step = regions.copy()
            step['year'] = year
            step['month'] = month
            records.append(step)
    master = pd.concat(records, ignore_index=True)
    master.rename(columns={'shapeName': 'admin2', 'shapeISO': 'country_iso'}, inplace=True)
    return master

print("Helper functions defined.")

Helper functions defined.


## 2. Load Raw Data

In [18]:
# Price data (WorldBank imputed)
price_df = pd.read_csv(
    os.path.join(DATA_DIR, 'worldbank_imputed_price_data/WLD_RTFP_mkt_2026-01-13.csv'),
    low_memory=False
)
price_df = price_df[price_df['ISO3'].isin(iso3_list)]
print(f"Price data: {len(price_df)} rows, {price_df['ISO3'].value_counts().to_dict()}")

# Crop mask (aggregated to admin2)
crop_df = pd.read_parquet(os.path.join(DATA_DIR, 'crop_mask/admin_mapped/admin_agg.parquet'))
print(f"Crop data: {len(crop_df)} rows")

# ACLED conflict data
acled_df = pd.read_excel(os.path.join(DATA_DIR, 'raw/acled/Africa_aggregated_data_up_to-2026-01-03.xlsx'))
acled_df = acled_df[acled_df['COUNTRY'].isin(['Kenya', 'Somalia'])]
print(f"ACLED data: {len(acled_df)} rows")

# FLDAS climate/hydrology data (v3)
fldas_path = os.path.join(DATA_DIR, 'fldas/processed/EastAfrica_FLDAS_Fixed_2007_2025.csv')
if os.path.exists(fldas_path):
    fldas_df = pd.read_csv(fldas_path)
    print(f"FLDAS data: {len(fldas_df)} rows, {len([c for c in fldas_df.columns if c not in ['year','month','country_iso','admin1','admin2']])} feature cols")
else:
    fldas_df = pd.DataFrame()
    print(f"FLDAS file not found: {fldas_path}")

# Vegetation Indices (v3)
veg_path = os.path.join(DATA_DIR, 'vegetation/processed/EastAfrica_Vegetation_Indices_2007_2025.csv')
if os.path.exists(veg_path):
    vegetation_df = pd.read_csv(veg_path)
    veg_feature_cols = [c for c in vegetation_df.columns if c not in ['year','month','country_iso','admin1','admin2']]
    print(f"Vegetation data: {len(vegetation_df)} rows, columns: {veg_feature_cols}")
else:
    vegetation_df = pd.DataFrame()
    print(f"Vegetation file not found: {veg_path}")

# Climate Indices (v3)
ci_path = os.path.join(DATA_DIR, 'climate_indices/processed/Climate_Indices_2007_2025.csv')
if os.path.exists(ci_path):
    climate_indices_df = pd.read_csv(ci_path)
    print(f"Climate indices: {len(climate_indices_df)} rows, columns: {list(climate_indices_df.columns)}")
else:
    climate_indices_df = pd.DataFrame()
    print(f"Climate indices file not found: {ci_path}")

Price data: 62288 rows, {'KEN': 51983, 'SOM': 10305}
Crop data: 1656 rows
ACLED data: 42136 rows
FLDAS data: 109896 rows, 33 feature cols
Vegetation data: 109896 rows, columns: ['NDVI', 'EVI', 'LST', 'VCI', 'TCI', 'VHI']
Climate indices: 228 rows, columns: ['year', 'month', 'NINO34_Anom', 'IOD_DMI', 'Western_V_Gradient', 'MEI_v2']


## 3. Load Canonical Boundaries

In [19]:
admin_gdf = load_canonical_boundaries(iso3_list, boundary_dir)
canonical_names = admin_gdf['shapeName'].unique().tolist()
print(f"Total admin2 regions: {len(canonical_names)}")
print(f"  KEN: {len(admin_gdf[admin_gdf['shapeISO']=='KEN'])}")
print(f"  SOM: {len(admin_gdf[admin_gdf['shapeISO']=='SOM'])}")

/var/folders/t7/ldqv1xt97rs4jyjvhxf4shgw0000gn/T/ipykernel_5439/616939592.py:43: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf['shapeISO'] = gdf['shapeISO'].replace('', np.nan).fillna(iso)
2026-03-11 14:40:26,821 - INFO - Loaded KEN: 290 admin2 regions
/var/folders/t7/ldqv1xt97rs4jyjvhxf4shgw0000gn/T/ipykernel_5439/616939592.py:43: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf['shapeISO'] = gdf['shapeISO'].replace('', np.nan).fillna(iso)
2026-03-11 14:40:26,855 - INFO - Loaded SOM: 118 admin2 regions


Total admin2 regions: 408
  KEN: 290
  SOM: 118


## 4. Process Price Data (Spatial Join — per country)

**Key fix (v2):** Spatial join is done per-country to prevent cross-boundary misassignment.  
In v1, SOM markets Doolow (4.16, 42.08) and Wadajir (5.00, 45.00) fell inside ETH polygons.

**Price columns:** `c_maize_fao` (available for both KEN & SOM), `c_food_price_index`, `c_sorghum`  
(v1 used `c_maize` which was empty for KEN)


In [20]:
# Per-country spatial join to prevent cross-boundary contamination
price_cols = ['c_maize_fao', 'c_food_price_index', 'c_sorghum']
price_joined_parts = []

for iso in iso3_list:
    iso_prices = price_df[price_df['ISO3'] == iso].dropna(subset=['lat', 'lon'])
    iso_bounds = admin_gdf[admin_gdf['shapeISO'] == iso]
    
    if iso_prices.empty:
        logger.warning(f"No price data with coordinates for {iso}")
        continue
    
    joined = spatial_join_points(iso_prices, iso_bounds, 'lon', 'lat')
    matched = joined['admin2_canonical'].notna().sum()
    unmatched = joined['admin2_canonical'].isna().sum()
    print(f"{iso}: {matched}/{len(joined)} matched ({matched/len(joined)*100:.1f}%), {unmatched} unmatched")
    
    # Show unmatched markets
    if unmatched > 0:
        unmatch_mkts = joined[joined['admin2_canonical'].isna()][['mkt_name','lat','lon']].drop_duplicates('mkt_name')
        print(f"  Unmatched: {unmatch_mkts['mkt_name'].tolist()}")
    
    price_joined_parts.append(joined)

price_joined = pd.concat(price_joined_parts, ignore_index=True)

# Aggregate: mean price per admin2/year/month
price_agg = price_joined.dropna(subset=['admin2_canonical']).groupby(
    ['year', 'month', 'admin2_canonical']
)[price_cols].mean().reset_index()

print(f"\nPrice aggregated: {len(price_agg)} rows")
print(f"Admin2 with price: {price_agg['admin2_canonical'].nunique()}")

KEN: 51296/51754 matched (99.1%), 458 unmatched
  Unmatched: ['Kipini', 'Vanga (Kwale)']
SOM: 9847/10076 matched (97.7%), 229 unmatched
  Unmatched: ['Wadajir']

Price aggregated: 25190 rows
Admin2 with price: 110


## 5. Process Population Data (WorldPop Zonal Stats)

In [22]:
pop_dfs = []
raster_map = {
    'KEN': 'ken_pop_2020_1km.tif',
    'SOM': 'som_pop_2020_1km.tif',
}

for iso in iso3_list:
    raster_path = os.path.join(DATA_DIR, 'population_worldpop', raster_map[iso])
    if os.path.exists(raster_path):
        iso_pop = process_worldpop_population(admin_gdf, raster_path, iso)
        if not iso_pop.empty:
            pop_dfs.append(iso_pop)
    else:
        logger.warning(f"Population raster not found: {raster_path}")

pop_agg = pd.concat(pop_dfs, ignore_index=True)
pop_agg = pop_agg.groupby('shapeName')['population'].sum().reset_index()
pop_agg.rename(columns={'shapeName': 'admin2_canonical'}, inplace=True)
print(f"Population: {len(pop_agg)} admin2 regions")

2026-03-11 14:43:14,694 - INFO - Processing WorldPop for KEN with 408 regions...
2026-03-11 14:43:15,626 - INFO - Processing WorldPop for SOM with 408 regions...


Population: 408 admin2 regions


## 6. Process Crop Data

In [23]:
crop_proc = crop_df[crop_df['shapeISO_ADM0'].isin(iso3_list)].copy()
canonical_set = set(canonical_names)

crop_proc['admin2_canonical'] = crop_proc['shapeName_ADM2'].where(
    crop_proc['shapeName_ADM2'].isin(canonical_set)
)

# Fuzzy match fallback for unmatched
unmatched_mask = crop_proc['admin2_canonical'].isna()
if unmatched_mask.any():
    fallback = fuzzy_match_names(crop_proc.loc[unmatched_mask, 'shapeName_ADM2'], canonical_names)
    crop_proc.loc[unmatched_mask, 'admin2_canonical'] = crop_proc.loc[unmatched_mask, 'shapeName_ADM2'].map(fallback)

crop_agg = crop_proc.dropna(subset=['admin2_canonical']).groupby(
    'admin2_canonical'
)['value'].mean().reset_index().rename(columns={'value': 'crop_cover_fraction'})

print(f"Crop data: {len(crop_agg)} admin2 regions")
print(f"Note: values are in % scale (1-83), NOT 0-1 fraction")

Crop data: 380 admin2 regions
Note: values are in % scale (1-83), NOT 0-1 fraction


## 7. Process ACLED Conflict Data

In [24]:
acled_joined = spatial_join_points(acled_df, admin_gdf, 'CENTROID_LONGITUDE', 'CENTROID_LATITUDE')

acled_joined['year'] = pd.to_datetime(acled_joined['WEEK']).dt.year
acled_joined['month'] = pd.to_datetime(acled_joined['WEEK']).dt.month

acled_agg = acled_joined.dropna(subset=['admin2_canonical']).groupby(
    ['year', 'month', 'admin2_canonical']
).agg({
    'FATALITIES': 'sum',
    'EVENTS': 'count'
}).reset_index().rename(columns={'EVENTS': 'conflict_events', 'FATALITIES': 'conflict_fatalities'})

print(f"ACLED aggregated: {len(acled_agg)} rows")
unmatched = acled_joined['admin2_canonical'].isna().sum()
print(f"Unmatched events: {unmatched} (mostly maritime)")

ACLED aggregated: 10105 rows
Unmatched events: 14 (mostly maritime)


## 8. Final Merge

In [25]:
# Create skeleton
years = sorted(price_df['year'].unique())
months = sorted(price_df['month'].unique())
master = create_master_skeleton(years, months, admin_gdf)
print(f"Skeleton: {master.shape} ({master['admin2'].nunique()} admin2 x {len(years)*len(months)} months)")

# Merge price
merged = pd.merge(master, price_agg,
    left_on=['year', 'month', 'admin2'], right_on=['year', 'month', 'admin2_canonical'], how='left')
merged.drop(columns=['admin2_canonical'], inplace=True, errors='ignore')

# Merge population
merged = pd.merge(merged, pop_agg, left_on='admin2', right_on='admin2_canonical', how='left')
merged.drop(columns=['admin2_canonical'], inplace=True, errors='ignore')

# Merge crop
merged = pd.merge(merged, crop_agg, left_on='admin2', right_on='admin2_canonical', how='left')
merged.drop(columns=['admin2_canonical'], inplace=True, errors='ignore')

# Merge ACLED
merged = pd.merge(merged, acled_agg,
    left_on=['year', 'month', 'admin2'], right_on=['year', 'month', 'admin2_canonical'], how='left')
merged.drop(columns=['admin2_canonical'], inplace=True, errors='ignore')

merged['conflict_events'] = merged['conflict_events'].fillna(0)
merged['conflict_fatalities'] = merged['conflict_fatalities'].fillna(0)

# ── v3: Merge FLDAS ──
if not fldas_df.empty:
    fldas_filtered = fldas_df[fldas_df['country_iso'].isin(iso3_list)].copy()
    fldas_cols = [c for c in fldas_filtered.columns if c not in ['year', 'month', 'country_iso', 'admin1', 'admin2']]
    fldas_merge = fldas_filtered[['year', 'month', 'country_iso', 'admin2'] + fldas_cols]
    before = merged.shape[1]
    merged = pd.merge(merged, fldas_merge,
                      on=['year', 'month', 'country_iso', 'admin2'], how='left')
    fldas_matched = merged[fldas_cols[0]].notna().sum()
    print(f"FLDAS: added {merged.shape[1] - before} columns, "
          f"{fldas_matched}/{len(merged)} rows matched ({fldas_matched/len(merged)*100:.1f}%)")
else:
    print("No FLDAS data. Skipping.")

# ── v3: Merge Vegetation Indices ──
if not vegetation_df.empty:
    veg_filtered = vegetation_df[vegetation_df['country_iso'].isin(iso3_list)].copy()
    veg_cols = [c for c in veg_filtered.columns if c not in ['year', 'month', 'country_iso', 'admin1', 'admin2']]
    veg_merge = veg_filtered[['year', 'month', 'country_iso', 'admin2'] + veg_cols]
    before = merged.shape[1]
    merged = pd.merge(merged, veg_merge,
                      on=['year', 'month', 'country_iso', 'admin2'], how='left')
    veg_matched = merged[veg_cols[0]].notna().sum()
    print(f"Vegetation: added {merged.shape[1] - before} columns, "
          f"{veg_matched}/{len(merged)} rows matched ({veg_matched/len(merged)*100:.1f}%)")
else:
    print("No Vegetation data. Skipping.")

# ── v3: Merge Climate Indices (global → broadcast to all admin2) ──
if not climate_indices_df.empty:
    ci_cols = [c for c in climate_indices_df.columns if c not in ['year', 'month']]
    ci_merge = climate_indices_df[['year', 'month'] + ci_cols].copy()
    before = merged.shape[1]
    merged = pd.merge(merged, ci_merge, on=['year', 'month'], how='left')
    ci_matched = merged[ci_cols[0]].notna().sum()
    print(f"Climate Indices: added {merged.shape[1] - before} columns, "
          f"{ci_matched}/{len(merged)} rows matched ({ci_matched/len(merged)*100:.1f}%)")
else:
    print("No Climate Indices data. Skipping.")

print(f"\nFinal merged: {merged.shape}")
print(f"Columns: {list(merged.columns)}")
merged.head()

Skeleton: (97920, 4) (408 admin2 x 240 months)
FLDAS: added 33 columns, 93024/97920 rows matched (95.0%)
Vegetation: added 6 columns, 93024/97920 rows matched (95.0%)
Climate Indices: added 4 columns, 93024/97920 rows matched (95.0%)

Final merged: (97920, 54)
Columns: ['admin2', 'country_iso', 'year', 'month', 'c_maize_fao', 'c_food_price_index', 'c_sorghum', 'population', 'crop_cover_fraction', 'conflict_fatalities', 'conflict_events', 'Tair_f_1mon_Z', 'RadT_1mon_Z', 'SoilMoi00_10cm_1mon_Z', 'SoilMoi10_40cm_1mon_Z', 'SoilMoi00_40cm_1mon_Z', 'Rainf_f_1mon_Z', 'Evap_1mon_Z', 'Qs_1mon_Z', 'Qsb_1mon_Z', 'Total_Runoff_1mon_Z', 'Water_Balance_1mon_Z', 'Tair_f_3mon_Z', 'RadT_3mon_Z', 'SoilMoi00_10cm_3mon_Z', 'SoilMoi10_40cm_3mon_Z', 'SoilMoi00_40cm_3mon_Z', 'Rainf_f_3mon_Z', 'Evap_3mon_Z', 'Qs_3mon_Z', 'Qsb_3mon_Z', 'Total_Runoff_3mon_Z', 'Water_Balance_3mon_Z', 'Tair_f_6mon_Z', 'RadT_6mon_Z', 'SoilMoi00_10cm_6mon_Z', 'SoilMoi10_40cm_6mon_Z', 'SoilMoi00_40cm_6mon_Z', 'Rainf_f_6mon_Z', 'Evap

,admin2,country_iso,year,month,c_maize_fao,c_food_price_index,c_sorghum,population,crop_cover_fraction,conflict_fatalities,...,NDVI,EVI,LST,VCI,TCI,VHI,NINO34_Anom,IOD_DMI,Western_V_Gradient,MEI_v2
0,Ainabkoi,KEN,2007,1,NaN,NaN,NaN,202832.859375,74.547631,0.0,...,0.649647,0.413600,25.426472,89.786492,83.030762,86.408627,0.764076,0.238247,-0.537076,0.64
1,Ainamoi,KEN,2007,1,NaN,NaN,NaN,201868.531250,64.759625,0.0,...,0.772079,0.487553,26.849736,91.111168,85.289975,88.200568,0.764076,0.238247,-0.537076,0.64
2,Aldai,KEN,2007,1,NaN,NaN,NaN,229172.781250,63.958435,0.0,...,0.763000,0.467977,26.630808,87.104311,87.815579,87.459945,0.764076,0.238247,-0.537076,0.64
3,Alego Usonga,KEN,2007,1,NaN,NaN,NaN,253480.312500,69.850509,0.0,...,0.748523,0.489513,28.352742,96.730957,96.820123,96.775532,0.764076,0.238247,-0.537076,0.64
4,Awendo,KEN,2007,1,NaN,NaN,NaN,135832.531250,70.101679,0.0,...,0.713528,0.454856,27.106111,86.021613,98.322049,92.171834,0.764076,0.238247,-0.537076,0.64


In [27]:
merged.isna().sum()/len(merged)

admin2                   0.000000
country_iso              0.000000
year                     0.000000
month                    0.000000
c_maize_fao              0.742749
c_food_price_index       0.742749
c_sorghum                0.742749
population               0.000000
crop_cover_fraction      0.068627
conflict_fatalities      0.000000
conflict_events          0.000000
Tair_f_1mon_Z            0.050000
RadT_1mon_Z              0.050000
SoilMoi00_10cm_1mon_Z    0.050000
SoilMoi10_40cm_1mon_Z    0.050000
SoilMoi00_40cm_1mon_Z    0.050000
Rainf_f_1mon_Z           0.050000
Evap_1mon_Z              0.050000
Qs_1mon_Z                0.050000
Qsb_1mon_Z               0.050000
Total_Runoff_1mon_Z      0.050000
Water_Balance_1mon_Z     0.050000
Tair_f_3mon_Z            0.050000
RadT_3mon_Z              0.050000
SoilMoi00_10cm_3mon_Z    0.050000
SoilMoi10_40cm_3mon_Z    0.050000
SoilMoi00_40cm_3mon_Z    0.050000
Rainf_f_3mon_Z           0.050000
Evap_3mon_Z              0.050000
Qs_3mon_Z     

## 9. Save

In [ ]:
output_path = os.path.join(DATA_DIR, 'processed/subnational_merged_v3_KEN_SOM.parquet')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged.to_parquet(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Shape: {merged.shape}")

---
# Quality Checks

Below are systematic checks on the merged data.


## QC 1: Null / Missing Values

In [28]:
null_counts = merged.isnull().sum()
null_pct = (merged.isnull().mean() * 100).round(2)
qc_null = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
print(qc_null)
print(f"\nTotal rows: {len(merged)}")

                       null_count  null_pct
admin2                          0      0.00
country_iso                     0      0.00
year                            0      0.00
month                           0      0.00
c_maize_fao                 72730     74.27
c_food_price_index          72730     74.27
c_sorghum                   72730     74.27
population                      0      0.00
crop_cover_fraction          6720      6.86
conflict_fatalities             0      0.00
conflict_events                 0      0.00
Tair_f_1mon_Z                4896      5.00
RadT_1mon_Z                  4896      5.00
SoilMoi00_10cm_1mon_Z        4896      5.00
SoilMoi10_40cm_1mon_Z        4896      5.00
SoilMoi00_40cm_1mon_Z        4896      5.00
Rainf_f_1mon_Z               4896      5.00
Evap_1mon_Z                  4896      5.00
Qs_1mon_Z                    4896      5.00
Qsb_1mon_Z                   4896      5.00
Total_Runoff_1mon_Z          4896      5.00
Water_Balance_1mon_Z         489

## QC 2: Admin2 Boundary Completeness

In [29]:
for iso in iso3_list:
    gdf = gpd.read_file(os.path.join(boundary_dir, f'gb_{iso}_ADM2.geojson'))
    expected = set(gdf['shapeName'].unique())
    actual = set(merged[merged['country_iso'] == iso]['admin2'].unique())
    missing = expected - actual
    extra = actual - expected
    print(f"{iso}: expected={len(expected)}, actual={len(actual)}, "
          f"missing={len(missing)}, extra={len(extra)}")
    if missing:
        print(f"  Missing: {sorted(missing)}")

KEN: expected=290, actual=290, missing=0, extra=0
SOM: expected=118, actual=118, missing=0, extra=0


## QC 3: Price Coverage per Country

In [30]:
for iso in iso3_list:
    sub = merged[merged['country_iso'] == iso]
    total_admin2 = sub['admin2'].nunique()
    for col in ['c_maize_fao', 'c_food_price_index', 'c_sorghum']:
        notnull = sub[col].notna().sum()
        admin2_with = sub[sub[col].notna()]['admin2'].nunique()
        print(f"{iso} {col}: {notnull}/{len(sub)} ({notnull/len(sub)*100:.1f}%), "
              f"{admin2_with}/{total_admin2} admin2")
    print()

KEN c_maize_fao: 15343/69600 (22.0%), 67/290 admin2
KEN c_food_price_index: 15343/69600 (22.0%), 67/290 admin2
KEN c_sorghum: 15343/69600 (22.0%), 67/290 admin2

SOM c_maize_fao: 9847/28320 (34.8%), 43/118 admin2
SOM c_food_price_index: 9847/28320 (34.8%), 43/118 admin2
SOM c_sorghum: 9847/28320 (34.8%), 43/118 admin2



## QC 4: Price Range & Outliers

In [31]:
for col in ['c_maize_fao', 'c_food_price_index', 'c_sorghum']:
    valid = merged[col].dropna()
    q1, q3 = valid.quantile(0.25), valid.quantile(0.75)
    iqr = q3 - q1
    n_outliers = ((valid < q1 - 3*iqr) | (valid > q3 + 3*iqr)).sum()
    print(f"{col}:")
    print(f"  Range: {valid.min():.2f} ~ {valid.max():.2f}")
    print(f"  Mean: {valid.mean():.2f}, Median: {valid.median():.2f}")
    print(f"  Negative: {(valid < 0).sum()}, Zero: {(valid == 0).sum()}")
    print(f"  Extreme outliers (3*IQR): {n_outliers}")
    print()

print("NOTE: c_maize_fao unit differs by country (KES vs SOS).")
print("Consider using c_food_price_index (normalized 0-2 scale) for cross-country modeling.")

c_maize_fao:
  Range: 975.00 ~ 69038.30
  Mean: 25126.41, Median: 24874.66
  Negative: 0, Zero: 0
  Extreme outliers (3*IQR): 0

c_food_price_index:
  Range: 0.21 ~ 1.95
  Mean: 0.97, Median: 0.92
  Negative: 0, Zero: 0
  Extreme outliers (3*IQR): 0

c_sorghum:
  Range: 856.80 ~ 65448.29
  Mean: 7591.74, Median: 4967.32
  Negative: 0, Zero: 0
  Extreme outliers (3*IQR): 900

NOTE: c_maize_fao unit differs by country (KES vs SOS).
Consider using c_food_price_index (normalized 0-2 scale) for cross-country modeling.


## QC 5: Cross-boundary Contamination Check

In [ ]:
# Verify no ETH data leaked in
print(f"ETH rows: {(merged['country_iso'] == 'ETH').sum()}")
print(f"Countries present: {merged['country_iso'].unique()}")
print(f"Duplicate (admin2, iso, year, month): {merged.duplicated(subset=['admin2','country_iso','year','month']).sum()}")

# Check admin2 names don't overlap between countries
cross = merged.groupby('admin2')['country_iso'].nunique()
shared = cross[cross > 1]
if len(shared):
    print(f"\nAdmin2 shared across countries: {list(shared.index)}")
else:
    print("\nNo admin2 names shared across countries.")

## QC 6: Skeleton Completeness

In [ ]:
yr_month_sizes = merged.groupby(['year', 'month']).size()
print(f"Year range: {merged['year'].min()} - {merged['year'].max()}")
print(f"Rows per year-month (should be constant): {yr_month_sizes.unique()}")

all_years = range(merged['year'].min(), merged['year'].max() + 1)
all_months = range(1, 13)
expected = set((y, m) for y in all_years for m in all_months)
actual = set(zip(merged['year'], merged['month']))
missing = expected - actual
if missing:
    print(f"Missing year-month combos: {sorted(missing)}")
else:
    print("All year-month combos present.")

## QC 7: Population / Crop / Conflict / FLDAS / Vegetation / Climate Indices

In [ ]:
print("=== Population ===")
print(f"  Null: {merged['population'].isnull().sum()}")
print(f"  Range: {merged['population'].min():.0f} ~ {merged['population'].max():.0f}")
print(f"  Time-invariant: {merged.groupby(['admin2','country_iso'])['population'].nunique().eq(1).all()}")

print("\n=== Crop Cover Fraction ===")
print(f"  Null: {merged['crop_cover_fraction'].isnull().sum()} ({merged['crop_cover_fraction'].isnull().mean()*100:.1f}%)")
print(f"  Range: {merged['crop_cover_fraction'].dropna().min():.2f} ~ {merged['crop_cover_fraction'].dropna().max():.2f}")
print(f"  NOTE: Values are in % (1-83), not 0-1 fraction")

# Crop null admin2
crop_null = merged[merged['crop_cover_fraction'].isnull()].groupby('country_iso')['admin2'].unique()
print(f"\n  Null admin2:")
for iso, admins in crop_null.items():
    print(f"    {iso} ({len(admins)}): {list(admins)}")

print("\n=== Conflict ===")
print(f"  Rows with conflict > 0: {(merged['conflict_events'] > 0).sum()} ({(merged['conflict_events'] > 0).mean()*100:.1f}%)")
for iso in iso3_list:
    sub = merged[merged['country_iso'] == iso]
    print(f"  {iso}: {(sub['conflict_events'] > 0).mean()*100:.1f}% rows have conflict")

# v3: FLDAS
fldas_cols_in_merged = [c for c in merged.columns if c.startswith(('Evap_', 'Rainf_', 'SoilMoi', 'Tair_', 'Wind_', 'Qair_', 'SWdown_', 'LWdown_', 'Psurf_'))]
if fldas_cols_in_merged:
    print(f"\n=== FLDAS ({len(fldas_cols_in_merged)} columns) ===")
    fldas_null_pct = merged[fldas_cols_in_merged].isnull().mean() * 100
    print(f"  Null %: min={fldas_null_pct.min():.1f}%, max={fldas_null_pct.max():.1f}%, mean={fldas_null_pct.mean():.1f}%")
    print(f"  Sample columns: {fldas_cols_in_merged[:5]}")

# v3: Vegetation
veg_cols_in_merged = [c for c in merged.columns if c in ['NDVI', 'EVI', 'LST_Day', 'LST_Night', 'VCI', 'TCI', 'VHI']]
if veg_cols_in_merged:
    print(f"\n=== Vegetation ({len(veg_cols_in_merged)} columns) ===")
    for col in veg_cols_in_merged:
        valid = merged[col].dropna()
        print(f"  {col}: null={merged[col].isnull().sum()} ({merged[col].isnull().mean()*100:.1f}%), "
              f"range={valid.min():.3f}~{valid.max():.3f}")

# v3: Climate Indices
ci_cols_in_merged = [c for c in merged.columns if c in ['NINO34', 'IOD_DMI', 'Western_V_Gradient', 'MEI_v2']]
if ci_cols_in_merged:
    print(f"\n=== Climate Indices ({len(ci_cols_in_merged)} columns) ===")
    for col in ci_cols_in_merged:
        valid = merged[col].dropna()
        print(f"  {col}: null={merged[col].isnull().sum()} ({merged[col].isnull().mean()*100:.1f}%), "
              f"range={valid.min():.3f}~{valid.max():.3f}")

## QC 8: KEN vs SOM Price Unit Comparison

In [ ]:
print("c_maize_fao is in LOCAL CURRENCY (KES for Kenya, SOS for Somalia)")
print("c_food_price_index is NORMALIZED (around 1.0)\n")

for col in ['c_maize_fao', 'c_food_price_index']:
    print(f"{col}:")
    for iso in iso3_list:
        v = merged[(merged['country_iso'] == iso) & (merged[col].notna())][col]
        if len(v) > 0:
            print(f"  {iso}: mean={v.mean():.2f}, median={v.median():.2f}, "
                  f"min={v.min():.2f}, max={v.max():.2f}")
    print()

## QC 9: Visual Checks (run these to inspect)

Run the cells below to generate maps and charts for visual inspection.


In [ ]:
# Price coverage map
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for idx, iso in enumerate(iso3_list):
    ax = axes[idx]
    gdf = gpd.read_file(os.path.join(boundary_dir, f'gb_{iso}_ADM2.geojson'))
    price_admins = set(merged[(merged['country_iso']==iso) & (merged['c_maize_fao'].notna())]['admin2'].unique())
    gdf['has_price'] = gdf['shapeName'].isin(price_admins)
    gdf.plot(column='has_price', ax=ax, legend=True, 
             cmap='RdYlGn', edgecolor='gray', linewidth=0.3,
             legend_kwds={'labels': ['No price data', 'Has price data']})
    ax.set_title(f'{iso}: Price Data Coverage ({len(price_admins)} admin2)')
    ax.axis('off')

plt.tight_layout()
plt.show()

# c_food_price_index time series (sample admin2)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for idx, iso in enumerate(iso3_list):
    ax = axes[idx]
    sub = merged[(merged['country_iso']==iso) & (merged['c_food_price_index'].notna())]
    sample_admins = sub['admin2'].unique()[:5]
    for admin2 in sample_admins:
        ts = sub[sub['admin2']==admin2].sort_values(['year','month'])
        ts['date'] = pd.to_datetime(ts[['year','month']].assign(day=1))
        ax.plot(ts['date'], ts['c_food_price_index'], label=admin2, alpha=0.7)
    ax.set_title(f'{iso}: Food Price Index (sample admin2)')
    ax.set_ylabel('c_food_price_index')
    ax.legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.show()